In [11]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch_geometric.loader import DataLoader

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset

In [12]:
model_global = torch.load("../models/gnn_checkpoint.pt", map_location=torch.device('cpu'))
model_cliff = torch.load("../models/gnn_model_clifford.pt", map_location=torch.device('cpu'))
model_haar = torch.load("../models/gnn_model_haar.pt", map_location=torch.device('cpu'))
model_quan = torch.load("../models/gnn_model_quansistor.pt", map_location=torch.device('cpu'))
model_rand = torch.load("../models/gnn_model_random.pt", map_location=torch.device('cpu'))

In [13]:
model_global

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0005,  0.0235, -0.0373,  ...,  0.0409,  0.0870, -0.2433],
                       [ 0.0617, -0.0206, -0.2062,  ...,  0.1419, -0.1107,  0.0150],
                       [-0.1005,  0.1833, -0.0209,  ...,  0.2241,  0.0860, -0.2587],
                       ...,
                       [-0.0251,  0.1399, -0.3376,  ...,  0.1401, -0.1835, -0.0181],
                       [-0.1884,  0.0726,  0.1752,  ...,  0.0680,  0.0324,  0.1162],
                       [-0.1461, -0.1813,  0.0248,  ...,  0.1236, -0.1774,  0.3059]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.0870, -0.0908, -0.1373, -0.1195, -0.0484,  0.1685, -0.0261,  0.1812,
                        0.1959,  0.0181, -0.1729,  0.1409,  0.0154, -0.1378,  0.1569, -0.0627,
                       -0.0193, -0.0368,  0.0887, -0.1961,  0.1848, -0.1056,  0.1888, -0.2046,
                        0.0560, -0.1984, -0.0283, -0.1532, -0.

In [14]:
model_cliff

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.2017, -0.2080, -0.0340,  ..., -0.1211, -0.1365,  0.1664],
                       [-0.1036,  0.0181, -0.0782,  ...,  0.0313, -0.0982, -0.1137],
                       [-0.0030,  0.0654,  0.0966,  ..., -0.0403,  0.0584, -0.1983],
                       ...,
                       [-0.0787,  0.1950,  0.0716,  ..., -0.0786, -0.1429, -0.1033],
                       [-0.2052,  0.0904,  0.2111,  ..., -0.0369, -0.1681,  0.1174],
                       [-0.1972, -0.1589,  0.0692,  ..., -0.1870, -0.0538, -0.0387]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.0295, -0.0468,  0.0793,  0.0736,  0.1086,  0.1937, -0.0895, -0.0992,
                       -0.0872,  0.1987, -0.0355,  0.1821, -0.1182,  0.0055, -0.0409,  0.1813,
                        0.0018,  0.1386,  0.0860,  0.1077, -0.0110,  0.1537,  0.0958,  0.0544,
                       -0.1861,  0.1857,  0.0456, -0.0964, -0.

In [15]:
model_quan

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.1155, -0.1625,  0.0443,  ...,  0.0765,  0.2087, -0.1491],
                       [ 0.1296,  0.0225,  0.1017,  ...,  0.0326, -0.2248, -0.1663],
                       [-0.0901, -0.0066, -0.1504,  ...,  0.0323,  0.0220, -0.0802],
                       ...,
                       [ 0.1882,  0.0989, -0.2029,  ..., -0.0835,  0.1779,  0.0121],
                       [-0.0385, -0.0699,  0.0300,  ..., -0.1420,  0.2110, -0.1837],
                       [-0.0985,  0.1516, -0.0085,  ..., -0.1263,  0.0270, -0.2006]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.1853, -0.1404, -0.1412, -0.1641,  0.1604,  0.0829, -0.0804,  0.2075,
                       -0.0777, -0.0527, -0.1548, -0.0690,  0.1410,  0.1130, -0.2022, -0.1102,
                        0.0920, -0.0622,  0.1331, -0.1309,  0.0913, -0.1002, -0.0483, -0.1956,
                        0.0637,  0.1026, -0.0370,  0.0700,  0.

In [16]:
model_rand

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.1200, -0.1518,  0.1079,  ..., -0.2370, -0.0982, -0.2525],
                       [ 0.0162, -0.1070,  0.1460,  ...,  0.0052, -0.0283,  0.1504],
                       [-0.0051, -0.1217, -0.1157,  ...,  0.0180, -0.1715,  0.1350],
                       ...,
                       [ 0.1037,  0.1086, -0.0638,  ..., -0.1067,  0.0792, -0.2031],
                       [ 0.2045, -0.1376, -0.1584,  ...,  0.0966, -0.1826,  0.1566],
                       [ 0.0770,  0.0691, -0.1477,  ...,  0.0068,  0.1386, -0.2205]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.1634, -0.1582, -0.0096,  0.0616,  0.1488, -0.0252,  0.1376, -0.1429,
                       -0.0102,  0.0822, -0.0214, -0.0345, -0.0296, -0.0154,  0.1867,  0.1070,
                        0.0718,  0.1076,  0.1599, -0.1795, -0.2049,  0.0580, -0.1484, -0.1098,
                        0.1792, -0.1938, -0.1108,  0.0261,  0.

In [17]:
model_haar

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0059, -0.2041,  0.0784,  ...,  0.0839,  0.2335, -0.2259],
                       [ 0.1921,  0.1337,  0.1211,  ...,  0.0465,  0.0517,  0.1535],
                       [ 0.0022, -0.0820,  0.0199,  ...,  0.0662,  0.1328,  0.0631],
                       ...,
                       [ 0.0232,  0.1059, -0.2006,  ...,  0.0907, -0.1133,  0.2236],
                       [ 0.1256, -0.0340, -0.1980,  ..., -0.1850,  0.2108,  0.1678],
                       [ 0.0136, -0.0077,  0.1596,  ..., -0.2179,  0.0231, -0.0292]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.1426,  0.1421, -0.0217, -0.0021,  0.0619,  0.1013,  0.0389, -0.1731,
                        0.1025,  0.0919,  0.1454,  0.1226,  0.1651, -0.1885,  0.0561,  0.0180,
                        0.1591,  0.0444,  0.0473,  0.1905, -0.1797, -0.0774,  0.1258,  0.0481,
                        0.0964, -0.0173, -0.0322, -0.2008,  0.

In [18]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    pred_root = d / "predictions"

    if family is not None:
        paths = sorted((pred_root / family).glob("*.pt"))
    else:
        paths = []
        if pred_root.exists():
            for family_dir in sorted(pred_root.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))

    if not paths:
        paths = sorted(d.glob("*.pt"))

    return [str(p.resolve()) for p in paths]


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    # Include full resolved paths in the digest to avoid collisions across folders/families.
    canonical = "|".join(sorted(str(Path(p).resolve()) for p in paths))
    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]
    tag = f"_{suffix}" if suffix else ""
    cache_dir = Path("..") / "qqe" / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return str(cache_dir.resolve())

In [19]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [20]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [33]:
family = "clifford"
global_feature_variant = "binned"
node_feature_backend_variant = None

In [40]:
MODEL_STATE_PATH = f"../models/gnn_model_{family}.pt"
CHECKPOINT_PATH = f"../models/gnn_model_{family}.pt"

In [35]:
pred_data_paths = collect_pred_paths("../outputs/data", family=family)
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 7000 .pt files for dataset.


In [36]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

In [41]:
# --------------------------------------------------
# 1. Build prediction dataset with training-compatible feature config
# --------------------------------------------------
if not pred_data_paths:
    raise RuntimeError("No prediction .pt files found. Check outputs/data/predictions/<family>.")

checkpoint = None
model_config = {}
fixed_all_gate_keys = None

if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        model_config = checkpoint.get("model_config", {}) or {}
        feature_config = checkpoint.get("feature_config", {}) or {}
        fixed_all_gate_keys = feature_config.get("all_gate_keys", None)

suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
root = _cache_root_for_paths(pred_data_paths, suffix=suffix)

pred_dataset = QuantumCircuitGraphDataset(
    root=root,
    pt_paths=pred_data_paths,
    global_feature_variant=global_feature_variant,
    node_feature_backend_variant=node_feature_backend_variant,
    fixed_all_gate_keys=fixed_all_gate_keys,
)

trained_node_in_dim = model_config.get("node_in_dim", None)
trained_global_in_dim = model_config.get("global_in_dim", None)

padded_pred_dataset = PaddedGraphDatasetWrapper(
    pred_dataset,
    target_node_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
    target_global_dim=trained_global_in_dim if trained_global_in_dim is not None else None,
    target_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
 )

pred_loader = DataLoader(
    padded_pred_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

node_in_dim = padded_pred_dataset.target_dim
global_in_dim = padded_pred_dataset.target_global_dim

print(f"Prediction graphs: {len(pred_dataset)}")
print(f"Prediction node feature dim: {node_in_dim}")
print(f"Prediction global feature dim: {global_in_dim}")
if trained_node_in_dim is not None or trained_global_in_dim is not None:
    print(f"Trained node/global dims from checkpoint: {trained_node_in_dim}/{trained_global_in_dim}")

Prediction graphs: 7000
Prediction node feature dim: 23
Prediction global feature dim: 7
Trained node/global dims from checkpoint: 23/7


In [48]:
pred_dataset[0]

Data(
  x=[227, 25],
  edge_index=[2, 276],
  y=[1],
  global_features=[1, 7],
  num_qubits=12,
  gate_counts={
    CNOT_count=61,
    H_count=32,
    I_count=34,
    S_count=56,
    T_count=20,
  },
  meta={
    cid='clifford_Q12_L11_S1204654993',
    family='clifford',
    n_qubits=12,
    n_layers=11,
    seed=1204654993,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [42]:
# --------------------------------------------------
# 2. Validate dataset/model dimensions
# --------------------------------------------------
print("node_in_dim (prediction) =", node_in_dim)
print("global_in_dim (prediction) =", global_in_dim)
print("node_in_dim (trained) =", trained_node_in_dim)
print("global_in_dim (trained) =", trained_global_in_dim)

if trained_node_in_dim is not None and node_in_dim != trained_node_in_dim:
    print(f"Node dim will be adapted to trained dim {trained_node_in_dim} at inference time.")

if trained_global_in_dim is not None and global_in_dim != trained_global_in_dim:
    print(f"Global feature dim will be adapted to trained dim {trained_global_in_dim} at inference time.")

node_in_dim (prediction) = 23
global_in_dim (prediction) = 7
node_in_dim (trained) = 23
global_in_dim (trained) = 7


In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [44]:
# --------------------------------------------------
# 3. Rebuild model and load weights robustly
# --------------------------------------------------
def _extract_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict) and "model_state_dict" in payload:
        return payload["model_state_dict"]
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise RuntimeError("Unsupported model file format.")

if checkpoint is not None and isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    if not Path(MODEL_STATE_PATH).exists():
        raise FileNotFoundError(f"Could not find model weights at {MODEL_STATE_PATH}")
    raw_payload = torch.load(MODEL_STATE_PATH, map_location="cpu")
    state_dict = _extract_state_dict(raw_payload)

model_kwargs = {
    "node_in_dim": int(trained_node_in_dim if trained_node_in_dim is not None else node_in_dim),
    "global_in_dim": int(trained_global_in_dim if trained_global_in_dim is not None else global_in_dim),
    "gnn_hidden": int(model_config.get("gnn_hidden", 32)),
    "gnn_heads": int(model_config.get("gnn_heads", 8)),
    "global_hidden": int(model_config.get("global_hidden", 16)),
    "reg_hidden": int(model_config.get("reg_hidden", 16)),
    "num_layers": int(model_config.get("num_layers", 5)),
    "dropout_rate": float(model_config.get("dropout_rate", 0.1)),
}

model = GNN(**model_kwargs).to(device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()

print("Loaded model config:", model_kwargs)
if missing_keys:
    print("Missing keys:", missing_keys)
if unexpected_keys:
    print("Unexpected keys:", unexpected_keys)

Loaded model config: {'node_in_dim': 23, 'global_in_dim': 7, 'gnn_hidden': 32, 'gnn_heads': 8, 'global_hidden': 16, 'reg_hidden': 16, 'num_layers': 5, 'dropout_rate': 0.1}


In [45]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [2.873539686203003, 2.7609477043151855, 2.8415908813476562, 2.757499933242798, 2.8242716789245605, 2.7501513957977295, 2.8204238414764404, 2.7312965393066406, 2.8272080421447754, 2.722076416015625]
Total predictions: 7000
